New version of drawflow. 

In [2]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array

In [3]:
#yield_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS"

#yield_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged/merged_yields.root"

In [4]:
yield_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60"

yield_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/merged_yields.root"

In [5]:
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [6]:
ROOT.gDirectory.Clear()

In [7]:

def openYields(filepath):

    wta_yields = {}
    std_yields = {}

    wta_avg_Nch = {}
    std_avg_Nch = {}

    file = ROOT.TFile.Open(filepath, "READ")

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
        
        hYield_wta = file.Get(f"WTA_yield_{mult_bin[0]}_{mult_bin[1]}")
        hYield_std = file.Get(f"STD_yield_{mult_bin[0]}_{mult_bin[1]}")

        hYield_wta.SetDirectory(0)
        hYield_std.SetDirectory(0)

        wta_yields[bin_key] = hYield_wta
        std_yields[bin_key] = hYield_std

        bin_wta_avg_Nch = file.Get(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        bin_std_avg_Nch = file.Get(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        wta_avg_Nch[bin_key] = np.float64(bin_wta_avg_Nch)
        std_avg_Nch[bin_key] = np.float64(bin_std_avg_Nch)

    return wta_yields, std_yields, wta_avg_Nch, std_avg_Nch

In [8]:


cosine_series = '''( [0]/(2*TMath::Pi()) )* (1 + 2* ( [1]*TMath::Cos(x) + [2]*TMath::Cos(2*x) 
+ [3]*TMath::Cos(3*x) + [4]*TMath::Cos(4*x) + [5]*TMath::Cos(5*x)))'''


def extract_fourier_harmonics(hYield_2D, mult_bin, cluster_type=""):
    """
    Applies the PRL paper methodology:
    1. Integrates over |delta_eta_star| > 2 (only positive side because yield is filled through 6-way symmetry).
    2. Decomposes the 1D projection into a Fourier cosine series via a TF1 fit.
    """
    xaxis = hYield_2D.GetXaxis()
    
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    # 1. Determine exact bin edges for the |delta_eta_star| > 2 condition
    # Positive side bins (2.0 < delta_eta_star <= 4.0)
    bin_pos_low = xaxis.FindBin(2.0 + 0.0001)  # Start strictly above 2.0
    bin_pos_high = hYield_2D.GetNbinsX()       # Go up to the maximum available bin
    
    # 2. Perform the projections on the Y-axis (delta_phi_star)
    h1D_pos = hYield_2D.ProjectionY(f"h1D_pos_{cluster_type}", bin_pos_low, bin_pos_high)
    
    # Combine both sides to satisfy the absolute value |delta_eta_star| > 2 requirement
    h1D_longrange = h1D_pos.Clone(f"h1D_longrange_{cluster_type}_{bin_name}")
    #h1D_longrange.Add(h1D_neg)
    
    # 3. Define the Fourier Cosine Series fit function (Eq. 4 of the paper)
    # [0] = N0 (Normalization parameter)
    # [1] = V1_delta
    # [2] = V2_delta
    # [3] = V3_delta
    #fit_formula = "[0] * (1.0 + 2.0*[1]*cos(x) + 2.0*[2]*cos(2*x) + 2.0*[3]*cos(3*x))"
    
    phi_min = h1D_longrange.GetXaxis().GetXmin()
    phi_max = h1D_longrange.GetXaxis().GetXmax()
    
    fit_func = ROOT.TF1(f"{cluster_type}_fourier_fit_{bin_name}", cosine_series, phi_min, phi_max)
    
    # Set intelligent initial parameter guesses to guide ROOT's Minuit minimizer
    fit_func.SetParameter(0, h1D_longrange.GetMaximum())   
    #fit_func.SetParameter(1, -0.05)                      # V1_delta is usually negative due to momentum conservation
    fit_func.SetParameter(1, 0.1)
    fit_func.SetParameter(2, 0.1)                         # V2_delta initial guess
    fit_func.SetParameter(3, 0.01)                        # V3_delta initial guess
    fit_func.SetParameter(4, 0.01)
    fit_func.SetParameter(5, 0.01)
    
    # Execute the fit over the full range
    h1D_longrange.Fit(fit_func, "m E q")
    
    # 4. Extract parameters and calculate v2* via factorization
    V2_delta = fit_func.GetParameter(2)
    V2_delta_err = fit_func.GetParError(2)
    
    if V2_delta > 0:
        v2_star = np.sqrt(V2_delta)
        # Error propagation for square root: sigma_v2 = sigma_V2 / (2 * sqrt(V2))
        v2_star_err = 0.5 * V2_delta_err / np.sqrt(V2_delta)
    else:
        # Handle cases where statistical fluctuations push V2 slightly negative
        v2_star = 0.0
        v2_star_err = 0.0
        
    print(f"--- Results for Multiplicity Bin {mult_bin} ---")
    print(f"{cluster_type} V2_delta = {V2_delta:.5f} ± {V2_delta_err:.5f}")
    print(f"{cluster_type} v2* = {v2_star:.5f} ± {v2_star_err:.5f}\n")
    
    return h1D_longrange, fit_func, [V2_delta, V2_delta_err], [v2_star, v2_star_err]

In [9]:
wta_yields, std_yields, wta_avg_Nch, std_avg_Nch = openYields(yield_file)

In [10]:

# for plotting fourier coeffs

wta_y_vals = []
wta_y_errs = []

std_y_vals = []
std_y_errs = []

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"

    h1D_wta, fit_wta, V2_delta_wta, v2_star_wta = extract_fourier_harmonics(wta_yields[bin_key], mult_bin, cluster_type="wta")
    h1D_std, fit_std, V2_delta_std, v2_star_std = extract_fourier_harmonics(std_yields[bin_key], mult_bin, cluster_type="std")

    wta_y_vals.append(V2_delta_wta[0])
    wta_y_errs.append(V2_delta_wta[1])

    std_y_vals.append(V2_delta_std[0])
    std_y_errs.append(V2_delta_std[1])
    #print(f"bin {bin_key} worked")
    



--- Results for Multiplicity Bin [0, 25] ---
wta V2_delta = 0.00018 ± 0.00115
wta v2* = 0.01356 ± 0.04223

--- Results for Multiplicity Bin [0, 25] ---
std V2_delta = 0.07865 ± 0.00181
std v2* = 0.28045 ± 0.00323

--- Results for Multiplicity Bin [25, 36] ---
wta V2_delta = 0.00247 ± 0.00037
wta v2* = 0.04971 ± 0.00371

--- Results for Multiplicity Bin [25, 36] ---
std V2_delta = 0.05125 ± 0.00073
std v2* = 0.22638 ± 0.00162

--- Results for Multiplicity Bin [36, 48] ---
wta V2_delta = 0.00167 ± 0.00084
wta v2* = 0.04085 ± 0.01026

--- Results for Multiplicity Bin [36, 48] ---
std V2_delta = 0.03669 ± 0.00097
std v2* = 0.19155 ± 0.00253

--- Results for Multiplicity Bin [48, 60] ---
wta V2_delta = 0.00572 ± 0.00172
wta v2* = 0.07563 ± 0.01139

--- Results for Multiplicity Bin [48, 60] ---
std V2_delta = 0.02226 ± 0.00030
std v2* = 0.14921 ± 0.00100

--- Results for Multiplicity Bin [60, 71] ---
wta V2_delta = -0.00221 ± 0.00094
wta v2* = 0.00000 ± 0.00000

--- Results for Multiplicity 

Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1


In [11]:
def draw_summary(wta_x_vals, std_x_vals, wta_y, wta_err, std_y, std_err):
    """
    Creates a single summary graph comparing WTA vs STD v2 values.
    """
    wta_n_points = len(wta_x_vals)
    std_n_points = len(std_x_vals)
    
    # CRITICAL: Convert standard Python lists to C-compatible double arrays for PyROOT
    wta_x = array.array('d', wta_x_vals)
    std_x = array.array('d', std_x_vals)
    y_wta = array.array('d', wta_y)
    ey_wta = array.array('d', wta_err)
    y_std = array.array('d', std_y)
    ey_std = array.array('d', std_err)
    
    # Initialize TGraphErrors for both configurations
    gr_wta = ROOT.TGraphErrors(wta_n_points, wta_x, y_wta, 0, ey_wta)
    gr_std = ROOT.TGraphErrors(std_n_points, std_x, y_std, 0, ey_std)
    
    # Canvas initialization & grid configuration matching the reference
    canvas = ROOT.TCanvas("c_summary", "Anisotropy Coefficients vs Multiplicity", 750, 650)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    canvas.SetGrid()
    
    # Style Winner-Take-All Data points (Filled red markers)
    gr_wta.SetMarkerStyle(20)
    gr_wta.SetMarkerColor(ROOT.kBlue)
    gr_wta.SetLineColor(ROOT.kBlue)
    gr_wta.SetLineWidth(2)
    gr_wta.SetMarkerSize(1.3)
    
    # Style Standard Axis Data points (Open blue markers)
    gr_std.SetMarkerStyle(25) # Open square
    gr_std.SetMarkerColor(ROOT.kRed)
    gr_std.SetLineColor(ROOT.kRed)
    gr_std.SetLineWidth(2)
    gr_std.SetMarkerSize(1.3)
    
    # Wrap both inside a TMultiGraph to cleanly coordinate unified x and y scaling limits
    mg = ROOT.TMultiGraph()
    mg.Add(gr_std, "P")
    mg.Add(gr_wta, "P")
    mg.SetTitle("; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
    mg.Draw("A") # "A" draws the bounding coordinate frame layout
    
    # Enforce strict axis scaling configurations
    mg.GetXaxis().SetLimits(0, 120)
    mg.GetHistogram().SetMinimum(-0.05)
    mg.GetHistogram().SetMaximum(0.15)
    
    mg.GetXaxis().SetTitleSize(0.045)
    mg.GetYaxis().SetTitleSize(0.045)
    mg.GetXaxis().SetLabelSize(0.04)
    mg.GetYaxis().SetLabelSize(0.04)
    
    # Render a baseline guide at v2 = 0
    line = ROOT.TLine(0, 0, 90, 0)
    line.SetLineStyle(2) # Dashed configuration
    line.SetLineColor(ROOT.kBlack)
    line.Draw()
    
    # Draw Legend using standard root bounding bounds
    legend = ROOT.TLegend(0.48, 0.74, 0.88, 0.88)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0) # Transparent background
    legend.SetTextSize(0.035)
    legend.AddEntry(gr_wta, "Winner-Take-All (WTA) Axis", "pe")
    legend.AddEntry(gr_std, "Standard Axis", "pe")
    legend.Draw()
    
    # Add standardized scientific annotations
    #latex = ROOT.TLatex()
    #latex.SetNDC()
    #latex.SetTextSize(0.033)
    #latex.DrawLatex(0.18, 0.42, "#bf{CMS} Pythia 8 Simulation")
    #latex.DrawLatex(0.18, 0.37, "pp collisions #sqrt{s} = 13 TeV")
    #latex.DrawLatex(0.18, 0.32, "Jet p_{T} > 550 GeV/c, |#eta_{jet}| < 1.6")
    #latex.DrawLatex(0.18, 0.27, "Associated 0.3 < j_{T} < 3.0 GeV/c")
    #latex.DrawLatex(0.18, 0.22, "2.0 < |#Delta#eta^{*}| < 4.0 (Long-range)")
    
    canvas.Update()
    canvas.SaveAs("v2_summary_plot_ROOT.pdf")
    canvas.Close()

In [13]:
draw_summary( list(wta_avg_Nch.values()), list(std_avg_Nch.values()),  wta_y_vals, wta_y_errs, std_y_vals, std_y_errs)

Info in <TCanvas::Print>: pdf file v2_summary_plot_ROOT.pdf has been created


In [14]:
print(list(wta_avg_Nch.values()))

[np.float64(21.774890875077947), np.float64(31.67633854686416), np.float64(42.647876805935184), np.float64(56.329228135908345), np.float64(63.384038800756116), np.float64(73.27218930654269), np.float64(81.50399829395344), np.float64(92.92930725722087), np.float64(101.56617556023276)]


In [15]:
print(list(std_avg_Nch.values()))

[np.float64(21.399123661148977), np.float64(31.399854048565867), np.float64(42.25002121326287), np.float64(57.876864153296005), np.float64(63.501380827915), np.float64(73.32442481092546), np.float64(81.63618613732508), np.float64(92.96314883194137), np.float64(101.73061183949146)]


# Combined graph

In [12]:
ROOT.gDirectory.Clear()

In [13]:
yield_file_0mb = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/merged_yields.root"
yield_file_3mb = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/3mb_nch60/merged/merged_yields.root"

In [14]:
wta_yields_0mb, std_yields_0mb, wta_avg_Nch_0mb, std_avg_Nch_0mb = openYields(yield_file_0mb)
wta_yields_3mb, std_yields_3mb, wta_avg_Nch_3mb, std_avg_Nch_3mb = openYields(yield_file_3mb)

In [15]:
# for plotting fourier coeffs

wta_y_vals_0mb = []
wta_y_errs_0mb = []
std_y_vals_0mb = []
std_y_errs_0mb = []

wta_y_errs_3mb = []
wta_y_vals_3mb = []
std_y_vals_3mb = []
std_y_errs_3mb = []

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"

    h1D_wta_0mb, fit_wta_0mb, V2_delta_wta_0mb, v2_star_wta_0mb = extract_fourier_harmonics(wta_yields_0mb[bin_key], mult_bin, cluster_type="wta")
    h1D_std_0mb, fit_std_0mb, V2_delta_std_0mb, v2_star_std_0mb = extract_fourier_harmonics(std_yields_0mb[bin_key], mult_bin, cluster_type="std")

    h1D_wta_3mb, fit_wta_3mb, V2_delta_wta_3mb, v2_star_wta_3mb = extract_fourier_harmonics(wta_yields_3mb[bin_key], mult_bin, cluster_type="wta")
    h1D_std_3mb, fit_std_3mb, V2_delta_std_3mb, v2_star_std_3mb = extract_fourier_harmonics(std_yields_3mb[bin_key], mult_bin, cluster_type="std")

    wta_y_vals_0mb.append(V2_delta_wta_0mb[0])
    wta_y_errs_0mb.append(V2_delta_wta_0mb[1])

    wta_y_vals_3mb.append(V2_delta_wta_3mb[0])
    wta_y_errs_3mb.append(V2_delta_wta_3mb[1])

    std_y_vals_0mb.append(V2_delta_std_0mb[0])
    std_y_errs_0mb.append(V2_delta_std_0mb[1])

    std_y_vals_3mb.append(V2_delta_std_3mb[0])
    std_y_errs_3mb.append(V2_delta_std_3mb[1])

--- Results for Multiplicity Bin [0, 25] ---
wta V2_delta = 0.00018 ± 0.00115
wta v2* = 0.01356 ± 0.04223

--- Results for Multiplicity Bin [0, 25] ---
std V2_delta = 0.07865 ± 0.00181
std v2* = 0.28045 ± 0.00323

--- Results for Multiplicity Bin [0, 25] ---
wta V2_delta = -0.00370 ± 0.00184
wta v2* = 0.00000 ± 0.00000

--- Results for Multiplicity Bin [0, 25] ---
std V2_delta = 0.08085 ± 0.00290
std v2* = 0.28434 ± 0.00509

--- Results for Multiplicity Bin [25, 36] ---
wta V2_delta = 0.00247 ± 0.00037
wta v2* = 0.04971 ± 0.00371

--- Results for Multiplicity Bin [25, 36] ---
std V2_delta = 0.05125 ± 0.00073
std v2* = 0.22638 ± 0.00162

--- Results for Multiplicity Bin [25, 36] ---
wta V2_delta = -0.00019 ± 0.00059
wta v2* = 0.00000 ± 0.00000

--- Results for Multiplicity Bin [25, 36] ---
std V2_delta = 0.04814 ± 0.00120
std v2* = 0.21940 ± 0.00274

--- Results for Multiplicity Bin [36, 48] ---
wta V2_delta = 0.00167 ± 0.00084
wta v2* = 0.04085 ± 0.01026

--- Results for Multiplicity B

In [16]:
# both 0mb and 3.0mb on the same graph


wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

# CRITICAL: Convert standard Python lists to C-compatible double arrays for PyROOT
wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)

# Initialize TGraphErrors for both configurations
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)

gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration matching the reference
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Winner-Take-All 0mb, open blue square
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColor(ROOT.kBlue)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

# Winner-take-all 3mb, closed blue circle
gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColor(ROOT.kBlue)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

# Standard Axis 0mb, open red square
gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColor(ROOT.kRed)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

# Standard Axis 3mb, closed red circle
gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColor(ROOT.kRed)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)

# Wrap both inside a TMultiGraph to cleanly coordinate unified x and y scaling limits
mg = ROOT.TMultiGraph()
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") # "A" draws the bounding coordinate frame layout

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.05)
mg.GetHistogram().SetMaximum(0.15)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 90, 0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Draw Legend using standard root bounding bounds
legend = ROOT.TLegend(0.48, 0.74, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) # Transparent background
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.Draw()


# Add standardized scientific annotations
#latex = ROOT.TLatex()
#latex.SetNDC()
#latex.SetTextSize(0.033)
#latex.DrawLatex(0.18, 0.42, "#bf{CMS} Pythia 8 Simulation")
#latex.DrawLatex(0.18, 0.37, "pp collisions #sqrt{s} = 13 TeV")
#latex.DrawLatex(0.18, 0.32, "Jet p_{T} > 550 GeV/c, |#eta_{jet}| < 1.6")
#latex.DrawLatex(0.18, 0.27, "Associated 0.3 < j_{T} < 3.0 GeV/c")
#latex.DrawLatex(0.18, 0.22, "2.0 < |#Delta#eta^{*}| < 4.0 (Long-range)")

canvas.Update()
canvas.SaveAs("combined_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_v2_summary_plot.pdf has been created


In [ ]:
# alternate version with grey bars



# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE GREY DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kGray, 0.8)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColor(ROOT.kGray)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kGray, 0.8)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColor(ROOT.kGray)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.74, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "0mb vs 3.0mb Difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt_v2_summary_plot.pdf has been created


In [16]:
print(type(gr_wta_boxes))

<class cppyy.gbl.TGraphErrors at 0x10cba7010>


In [24]:
# alternate version with colored bars



# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE GREY DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt2_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt2_v2_summary_plot.pdf has been created


In [17]:

# same but boxes are in between

# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt3_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt3_v2_summary_plot.pdf has been created


In [ ]:
# 4th alternate version with colored bars, and lines connecting points, and plotting middle of the bars


# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt3_v2_summary_plot.pdf")
canvas.Close()

In [18]:
import array
import ROOT

# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# GENERATE THE DIFFERENCE BOXES
# =====================================================================
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)


# =====================================================================
# INITIALIZE & STYLE GRAPH DATASETS (WITH DASHED LINES)
# =====================================================================
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# WTA 0mb: Open Squares, Blue, Long Dashed Line
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetLineStyle(7)  # Style 7 = Long Dash
gr_wta_0mb.SetMarkerSize(1.3)

# WTA 3.0mb: Closed Circles, Blue, Short Dashed Line
gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetLineStyle(2)  # Style 2 = Short Dash
gr_wta_3mb.SetMarkerSize(1.3)

# Standard 0mb: Open Squares, Red, Long Dashed Line
gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetLineStyle(7)  # Style 7 = Long Dash
gr_std_0mb.SetMarkerSize(1.3)

# Standard 3mb: Closed Circles, Red, Short Dashed Line
gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetLineStyle(2)  # Style 2 = Short Dash
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# ADJUSTED LAYER MANAGEMENT FOR SMOOTH LINES
# =====================================================================
mg = ROOT.TMultiGraph()

# Layer 1: Background Bars
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Layer 2: Smooth Connecting Curves ("C" option)
mg.Add(gr_wta_0mb, "C")
mg.Add(gr_std_0mb, "C")
mg.Add(gr_wta_3mb, "C")
mg.Add(gr_std_3mb, "C")

# Layer 3: Foreground Markers ("P" option)
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce scaling limits
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Baseline guide
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup - changed draw option to "lpe" to show the distinct line dashes
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "lpe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "lpe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "lpe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "lpe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt3_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt3_v2_summary_plot.pdf has been created
